# Generate one year of noisy LISA data

Use the simulator already in this repository: **ESA orbits → LISA Instrument → PyTDI → XYZ/AET**.
The default is **instrument noise only**. An optional binary injection and an existing GW-response-file mode are provided.

**Start here:** select the **Python (lisa-tf)** kernel, run through the short pilot, inspect its plots,
then execute the full-year cell. The full-year cell starts a substantial local simulation.

The default science interval is one **Julian year = 365.25 days = 31,557,600 s**. It contains
**7,889,400 samples at 4 s spacing**, with the final endpoint excluded. Change the duration to
`365 * 86400.0` if you mean a 365-day year.

The instrument is simulated continuously at 0.25 s measurement spacing (internal physics spacing 0.0625 s).
TDI is computed at that rate and then filtered and decimated to 4 s; instrument measurements are not decimated before TDI.
Native scheduler chunks preserve the noise realization. Overlapping TDI blocks retain the same continuous time series.

Outputs are **frequency combinations in Hz**, not equivalent strain. This uses the repository's TCB/light-travel-time
convention, second-generation Michelson combinations, and
$A=(Z-X)/\sqrt{2}$, $E=(X-2Y+Z)/\sqrt{6}$, $T=(X+Y+Z)/\sqrt{3}$.
All default instrumental noises remain enabled. The existing Michelson evaluator does **not** perform sideband clock-noise
calibration. Galactic foregrounds, a source population, gaps and injected glitches are not included by default.

## 1. Environment and repository

The local `lisa-tf` environment already contains the needed packages. From a new environment, install the simulation
requirements before starting the kernel; this notebook does not install packages automatically. A minimal setup is:

```bash
python -m pip install numpy scipy pandas h5py matplotlib ipykernel nbformat \
  lisaorbits==3.0.3 lisagwresponse==3.0.2 lisainstrument==2.1.0 pytdi==2.1.0
```

Run from this checkout or its `noise only generation/` directory. The helper calls the existing simulation's TDI evaluator.

In [ ]:
from pathlib import Path
from dataclasses import asdict, replace
from datetime import datetime, timezone
import importlib.metadata as metadata
import json
import logging
import os
import shutil
import subprocess
import sys
import time
import uuid
import warnings

PROJECT_ROOT = next(
    (p for p in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
     if (p / "pipeline/research/scripts/six_link_generation.py").is_file()),
    None,
)
if PROJECT_ROOT is None:
    raise RuntimeError("Open this notebook from the cWB-space checkout or its noise only generation directory.")
NOTEBOOK_DIR = PROJECT_ROOT / "noise only generation"
for folder in (PROJECT_ROOT, NOTEBOOK_DIR):
    if str(folder) not in sys.path:
        sys.path.insert(0, str(folder))

os.environ.setdefault("MPLCONFIGDIR", str(PROJECT_ROOT / "outputs/.mplconfig"))
Path(os.environ["MPLCONFIGDIR"]).mkdir(parents=True, exist_ok=True)

import h5py
import matplotlib.pyplot as plt
import numpy as np
from scipy.signal import welch
from lisaorbits import OEMOrbits
from lisagwresponse import GalacticBinary
from lisainstrument import Instrument, SchedulerConfigSerial
from lisainstrument.instru import SimResultsNumpyCore
from lisa_year_helpers import (
    CHANNELS, JULIAN_YEAR_SECONDS, YearPlan, antialias_filter, processing_margin,
    compare_pilot_to_full_record, sample_count, sha256_file,
    write_binary_response, write_json, write_tdi_product,
)

VERSIONS = {name: metadata.version(name) for name in (
    "numpy", "scipy", "pandas", "h5py", "matplotlib", "lisaorbits", "lisagwresponse", "lisainstrument", "pytdi"
)}
for name, expected in {"lisaorbits": "3.0.3", "lisagwresponse": "3.0.2", "lisainstrument": "2.1.0", "pytdi": "2.1.0"}.items():
    if VERSIONS[name] != expected:
        warnings.warn(f"{name} {VERSIONS[name]} differs from the notebook's tested version {expected}; recheck the pilot.")
logging.basicConfig(level=logging.WARNING)
print("Kernel:", sys.executable)
print(json.dumps(VERSIONS, indent=2))

## 2. Settings

Keep `GW_MODE = "none"` for noise only. Set `"binary"` to inject the explicitly specified 3 mHz test binary.
Set `"file"` to inject a LISA GW-response HDF5 file produced by the simulator, and supply its absolute path.
A strain NPZ is **not** a link-response file. File-mode input must use compatible orbits and absolute TCB times,
contain `tcb/y` in the standard link order `(12, 23, 31, 13, 32, 21)` (receiver first), and cover the full padded interval.
The coverage check below rejects short or misaligned files.

The $10^{-5}$ Hz noise cutoff is an **explicit low-frequency regularization**, matching the short simulator examples.
A year of data does not imply a trustworthy noise model down to $1/\mathrm{year}$.
A different low-frequency model needs a separate choice of cutoff, initialization and validation.
The output antialiasing filter has a 0.1 Hz passband edge and a 0.125 Hz stopband edge; there is no extra FFT bandpass.

In [ ]:
DURATION_SECONDS = JULIAN_YEAR_SECONDS
MEASUREMENT_DT_S = 0.25
OUTPUT_DT_S = 4.0
SEED = 20261029
NOISE_FMIN_HZ = 1e-5
CENTRAL_FREQ_HZ = 2.816e14
ORBIT_NAME = "esa-trailing"
ORBIT_VERSION = "2.0"       # explicit ESA ephemeris version, not "latest"
ORBIT_DT_S = 600.0
SCHEDULER_CHUNK_SIZE = 250_000  # internal scheduler units, not independent noise segments
GW_MODE = "none"            # "none", "binary", or "file"
EXISTING_GW_H5 = None        # e.g. Path("/absolute/path/to/gws.h5")
BINARY = dict(A=1e-21, f=3e-3, df=0.0, phi0=0.0, iota=0.7, psi=0.2, ra=np.pi / 4, dec=0.3)

plan = YearPlan(
    duration_s=DURATION_SECONDS,
    measurement_dt_s=MEASUREMENT_DT_S,
    output_dt_s=OUTPUT_DT_S,
    guard_s=2048.0,
    tdi_block_s=86400.0,
    passband_hz=0.1,
)
# A fresh directory for this configuration. Existing results are never overwritten.
run_id = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ") + "_" + uuid.uuid4().hex[:6]
OUTPUT_ROOT = PROJECT_ROOT / "outputs" / f"lisa_noise_{run_id}"
print(f"Science: {plan.duration_s / 86400:g} days; {plan.output_size:,} samples/channel")
print(f"Output Nyquist: {0.5 / plan.output_dt_s:g} Hz; nominal Fourier spacing: {1 / plan.duration_s:.9g} Hz")
print("New run root:", OUTPUT_ROOT)

## 3. Resource preflight

The final eight arrays (two time coordinates and six TDI channels) require about **0.47 GiB** before compression.
The complete instrument file is much larger. The figure below is a conservative uncompressed upper estimate
that treats even constant and telemetry quantities as full-rate arrays; actual storage can be smaller.
It is not a measured memory or runtime estimate. The short pilot reports its actual file size and elapsed time.

Memory is controlled by the native instrument scheduler and by the TDI block size. Changing the measurement
spacing or filter settings requires renewed numerical/sampling checks. Keep the default rate for the tested route.
The preflight checks disk space on the output filesystem and never removes an existing product.

In [ ]:
def resource_preflight(candidate, directory):
    # Core quantities are at measurement or slower cadence. Counting each at
    # full cadence overestimates storage, including constants stored as scalars.
    n_core = len(SimResultsNumpyCore.dataset_identifier_set())
    instrument_upper = n_core * candidate.instrument_size * 8
    product_bytes = 8 * candidate.output_size * 8
    gw_bytes = (candidate.instrument_size + 2 * candidate.guard_size) * 6 * 8 if GW_MODE == "binary" else 0
    needed = int(1.2 * (instrument_upper + product_bytes + gw_bytes) + 5 * 1024**3)
    ancestor = Path(directory).resolve()
    while not ancestor.exists():
        ancestor = ancestor.parent
    available = shutil.disk_usage(ancestor).free
    report = dict(
        instrument_upper_GiB=instrument_upper / 1024**3,
        final_arrays_GiB=product_bytes / 1024**3,
        optional_GW_GiB=gw_bytes / 1024**3,
        required_with_reserve_GiB=needed / 1024**3,
        available_GiB=available / 1024**3,
        output_samples=candidate.output_size,
        instrument_samples=candidate.instrument_size,
        fir_taps=len(antialias_filter(candidate)),
        processing_margin_s=processing_margin(candidate, antialias_filter(candidate)) * candidate.measurement_dt_s,
    )
    print(json.dumps(report, indent=2))
    if needed > available:
        raise OSError("Insufficient disk space for this conservative plan. Choose another OUTPUT_ROOT or change the plan.")
    return report

year_resources = resource_preflight(plan, OUTPUT_ROOT)

## 4. Generator

This function creates a fresh orbit file covering the requested interval and margins, then runs **one**
continuous instrument simulation with a fixed seed. It does not join independent day-long simulations.
The ESA orbit files may be downloaded by `lisaorbits` on first use; cached files are reused and their hashes recorded.
The exact science epoch is printed and stored rather than silently assigning the first sample to zero mission time.

TDI is evaluated in overlapping blocks of the continuous measurements. The retained interval excludes the
record's interpolation/filter guards. Symmetric filtering compensates its own group delay. A/E/T are
formed from the filtered XYZ arrays: these linear operations commute, and this order avoids separate
filter-roundoff errors in the saved A/E/T identities. There is no per-block FFT cutoff or arbitrary alignment shift. The manifest records configuration, effective instrument
metadata, software/code versions, source mode, hashes, file sizes and completion states.

A failed stage leaves an incomplete receipt and/or `.partial.h5` file. Start a fresh run directory after fixing
an error; an incomplete file is never accepted as a successful dataset. The notebook does not implement restart recovery.

In [ ]:
def generate_dataset(run_dir, candidate):
    if GW_MODE not in {"none", "binary", "file"}:
        raise ValueError("GW_MODE must be none, binary or file.")
    if not 0 < NOISE_FMIN_HZ < candidate.passband_hz:
        raise ValueError("Noise cutoff must be positive and below the science passband.")
    if GW_MODE == "file" and (EXISTING_GW_H5 is None or not Path(EXISTING_GW_H5).is_file()):
        raise FileNotFoundError("Set EXISTING_GW_H5 to a compatible full-duration GW-response file.")
    if GW_MODE == "binary":
        endpoint_frequencies = (
            BINARY["f"] - 2 * candidate.guard_s * BINARY["df"],
            BINARY["f"] + (candidate.duration_s + 2 * candidate.guard_s) * BINARY["df"],
        )
        if not 0 < min(endpoint_frequencies) <= max(endpoint_frequencies) < candidate.passband_hz:
            raise ValueError("The test binary must remain inside the selected passband, including margins.")
    run_dir = Path(run_dir)
    budget = resource_preflight(candidate, run_dir)
    run_dir.mkdir(parents=True, exist_ok=False)
    receipt_path = run_dir / "manifest.json"
    code_files = [
        NOTEBOOK_DIR / "generate_one_year_noisy_lisa.ipynb",
        NOTEBOOK_DIR / "lisa_year_helpers.py",
        PROJECT_ROOT / "pipeline/research/scripts/six_link_generation.py",
        PROJECT_ROOT / "pipeline/research/scripts/tdi_computation.py",
    ]
    revision = subprocess.run(["git", "rev-parse", "HEAD"], cwd=PROJECT_ROOT, capture_output=True, text=True)
    status = subprocess.run(["git", "status", "--porcelain"], cwd=PROJECT_ROOT, capture_output=True, text=True)
    receipt = dict(
        status="STARTED", created_utc=datetime.now(timezone.utc).isoformat(),
        plan=asdict(candidate), seed=SEED, noise_fmin_hz=NOISE_FMIN_HZ,
        central_freq_hz=CENTRAL_FREQ_HZ, versions=VERSIONS,
        orbit_name=ORBIT_NAME, orbit_version=ORBIT_VERSION, orbit_dt_s=ORBIT_DT_S,
        orbit_dataset="tcb/ltt", gw_dataset="tcb/y", gw_mode=GW_MODE,
        binary=BINARY if GW_MODE == "binary" else None,
        git_revision=revision.stdout.strip(), git_status=status.stdout.strip(),
        code_sha256={str(p.relative_to(PROJECT_ROOT)): sha256_file(p) for p in code_files},
        resources=budget, scheduler_chunk_size=SCHEDULER_CHUNK_SIZE,
        scope="Default instrumental noises; uncalibrated-clock Michelson TDI; no foreground population or gaps.",
    )
    write_json(receipt_path, receipt)
    started = time.monotonic()
    try:
        day = 86400.0
        oem = OEMOrbits.from_included(ORBIT_NAME, version=ORBIT_VERSION, extrapolate=False)
        orbit_t0 = np.ceil((oem.t_init + day) / ORBIT_DT_S) * ORBIT_DT_S
        instrument_t0 = float(orbit_t0 + 2 * day)
        science_t0 = instrument_t0 + candidate.guard_s
        raw_duration = candidate.instrument_size * candidate.measurement_dt_s
        orbit_size = int(np.ceil((4 * day + raw_duration) / ORBIT_DT_S)) + 1
        orbit_stop = orbit_t0 + (orbit_size - 1) * ORBIT_DT_S
        if orbit_t0 < oem.t_start + 60 or orbit_stop > oem.t_end - 60:
            raise ValueError("Requested science interval exceeds the supported ESA ephemeris span.")
        orbit_file = run_dir / "orbits.h5"
        oem.write(str(orbit_file), t0=float(orbit_t0), dt=ORBIT_DT_S, size=orbit_size, mode="x")
        receipt["orbit_sha256"] = sha256_file(orbit_file)
        receipt["oem_sha256"] = {str(p): sha256_file(p) for p in (oem.filename_1, oem.filename_2, oem.filename_3)}
        receipt.update(instrument_t0_tcb_s=instrument_t0, science_t0_tcb_s=science_t0)
        print(f"Science TCB interval: [{science_t0:.6f}, {science_t0 + candidate.duration_s:.6f}) s since LISA epoch", flush=True)

        gw_file = None
        # Extra response coverage handles the instrument's interpolation/filter history.
        gw_t0 = instrument_t0 - candidate.guard_s
        gw_size = candidate.instrument_size + 2 * candidate.guard_size
        if GW_MODE == "binary":
            source = GalacticBinary(orbits=str(orbit_file), t_init=science_t0, **BINARY)
            gw_file = run_dir / "gws.h5"
            write_binary_response(source, gw_file, t0=gw_t0, size=gw_size, dt=candidate.measurement_dt_s)
        elif GW_MODE == "file":
            gw_file = Path(EXISTING_GW_H5).resolve()
        if gw_file is not None:
            with h5py.File(gw_file, "r") as gw:
                first = float(gw.attrs["t0"])
                last = first + (int(gw.attrs["size"]) - 1) * float(gw.attrs["dt"])
                required_last = gw_t0 + (gw_size - 1) * candidate.measurement_dt_s
                if "tcb/y" not in gw or gw["tcb/y"].shape != (int(gw.attrs["size"]), 6):
                    raise ValueError("GW file must contain the six standard fractional-frequency responses in tcb/y.")
                if first > gw_t0 + 1e-7 or last < required_last - 1e-7:
                    raise ValueError("GW file does not cover the entire padded absolute TCB interval.")
            receipt.update(gw_file=str(gw_file), gw_sha256=sha256_file(gw_file))

        instrument = Instrument(
            size=candidate.instrument_size, dt=candidate.measurement_dt_s, t0=instrument_t0,
            orbits=str(orbit_file), orbit_dataset="tcb/ltt",
            gws=None if gw_file is None else str(gw_file), chunk_gw_files=True,
            central_freq=CENTRAL_FREQ_HZ, seed=SEED, physics_upsampling=4,
            # At the default 0.25 s spacing this is exactly the package default.
            aafilter=("kaiser", 240.0, 0.275 / candidate.measurement_dt_s, 0.725 / candidate.measurement_dt_s),
            noises_f_min_hz=NOISE_FMIN_HZ, clock_f_min_hz=NOISE_FMIN_HZ,
        )
        receipt["status"] = "GENERATING_INSTRUMENT"
        write_json(receipt_path, receipt)
        partial_instrument = run_dir / "instrument.partial.h5"
        instrument_file = run_dir / "instrument.h5"
        print(f"Continuous instrument export: {candidate.instrument_size:,} measurement samples", flush=True)
        instrument.export_hdf5(
            path=str(partial_instrument), overwrite=False, keep_all=False,
            cfgscheduler=SchedulerConfigSerial(chunk_size=SCHEDULER_CHUNK_SIZE),
        )
        partial_instrument.rename(instrument_file)
        del instrument
        with h5py.File(instrument_file, "r") as raw:
            receipt["instrument_effective_metadata"] = json.loads(raw.attrs["metadata_json"])
        receipt.update(status="GENERATING_TDI", instrument_bytes=instrument_file.stat().st_size)
        write_json(receipt_path, receipt)
        tdi_file = run_dir / "tdi_XYZ_AET.h5"
        diagnostics = write_tdi_product(instrument_file, tdi_file, candidate, central_freq_hz=CENTRAL_FREQ_HZ)
        receipt.update(
            status="COMPLETE", elapsed_s=time.monotonic() - started,
            diagnostics=diagnostics, tdi_bytes=tdi_file.stat().st_size,
            tdi_sha256=sha256_file(tdi_file),
        )
        write_json(receipt_path, receipt)
        print(f"Completed in {receipt['elapsed_s']:.1f} s; instrument {receipt['instrument_bytes'] / 1024**3:.3f} GiB; TDI {receipt['tdi_bytes'] / 1024**2:.2f} MiB", flush=True)
        return dict(directory=run_dir, instrument=instrument_file, tdi=tdi_file, manifest=receipt_path)
    except Exception as error:
        receipt.update(status="FAILED", error=f"{type(error).__name__}: {error}", elapsed_s=time.monotonic() - started)
        write_json(receipt_path, receipt)
        raise

## 5. Short pilot first

This 4096 s run checks the actual noise → measurements → TDI → saved-file route, including multiple TDI blocks.
It compares the overlapping-block result with the existing whole-record PyTDI route, with a relative L2 tolerance
of $10^{-9}$ per channel, and saves the measured errors in `pilot/numerical_checks.json`.
It cannot validate the noise spectrum near $10^{-5}$ Hz or annual orbital effects.
The pilot and year use the selected source mode and seed, but are separate simulations; the pilot is not a prefix
that gets appended to the year.

In [ ]:
pilot_plan = replace(plan, duration_s=4096.0, tdi_block_s=512.0)
pilot = generate_dataset(OUTPUT_ROOT / "pilot", pilot_plan)
pilot

In [ ]:
def inspect_product(product, candidate):
    max_aet_residual = 0.0
    xyz_scale = 0.0
    with h5py.File(product["tdi"], "r") as data:
        assert data.attrs["status"] == "COMPLETE"
        assert data.attrs["size"] == candidate.output_size
        for name in ("t", "t_lisa", *CHANNELS):
            assert data[name].shape == (candidate.output_size,), name
        for start in range(0, candidate.output_size, 100_000):
            stop = min(start + 100_000, candidate.output_size)
            block = {name: data[name][start:stop] for name in ("t", "t_lisa", *CHANNELS)}
            if any(not np.isfinite(v).all() for v in block.values()):
                raise ValueError("Non-finite output samples.")
            expected_t = np.arange(start, stop, dtype=float) * candidate.output_dt_s
            np.testing.assert_allclose(block["t"], expected_t, rtol=0, atol=1e-9)
            np.testing.assert_allclose(block["t_lisa"], data.attrs["t0"] + expected_t, rtol=0, atol=1e-7)
            x, y, z = (block[name] for name in ("X2", "Y2", "Z2"))
            expected = ((z - x) / np.sqrt(2), (x - 2*y + z) / np.sqrt(6), (x + y + z) / np.sqrt(3))
            max_aet_residual = max(max_aet_residual, *(float(np.max(np.abs(block[name] - value))) for name, value in zip(("A2", "E2", "T2"), expected)))
            xyz_scale = max(xyz_scale, *(float(np.max(np.abs(v))) for v in (x, y, z)))
        assert data["t"][0] == 0.0
        np.testing.assert_allclose(data["t"][-1] + candidate.output_dt_s, candidate.duration_s, rtol=0, atol=1e-7)
    if xyz_scale == 0 or max_aet_residual > 5e-12 * xyz_scale:
        raise ValueError("A/E/T linear-combination check failed or all XYZ samples are zero.")
    report = dict(samples=candidate.output_size, duration_s=candidate.duration_s, max_aet_residual_over_xyz=max_aet_residual / xyz_scale)
    print(json.dumps(report, indent=2))
    return report

pilot_qc = inspect_product(pilot, pilot_plan)
pilot_reference = compare_pilot_to_full_record(pilot["instrument"], pilot["tdi"], pilot_plan)
write_json(pilot["directory"] / "numerical_checks.json", dict(array_qc=pilot_qc, full_record_comparison=pilot_reference))
print(json.dumps(pilot_reference, indent=2))

In [ ]:
def plot_product(product, candidate, label):
    # Read at most seven days. This is an empirical Welch ASD in Hz/sqrt(Hz),
    # not a theoretical sensitivity curve or a calibrated noise-floor test.
    with h5py.File(product["tdi"], "r") as data:
        count = min(candidate.output_size, int(7 * 86400 / candidate.output_dt_s))
        times = data["t"][:count]
        channels = {name: data[name][:count] for name in ("A2", "E2", "T2")}
    fig, axes = plt.subplots(1, 2, figsize=(12, 3.8))
    stride = max(1, count // 10_000)  # display thinning only
    for name, values in channels.items():
        axes[0].plot(times[::stride] / 86400, values[::stride], label=name, lw=0.6, alpha=0.8)
        f, psd = welch(values, fs=1 / candidate.output_dt_s, nperseg=min(count, 2**17), detrend="constant", scaling="density")
        keep = (f > 0) & (f <= candidate.passband_hz)
        axes[1].loglog(f[keep], np.sqrt(psd[keep]), label=name)
    axes[0].set(xlabel="Time since science start [day]", ylabel="TDI frequency combination [Hz]", title=label)
    axes[1].set(xlabel="Frequency [Hz]", ylabel="One-sided ASD [Hz / sqrt(Hz)]", title="Empirical Welch ASD")
    axes[1].axvline(NOISE_FMIN_HZ, color="k", ls=":", alpha=0.5, label="noise regularization cutoff")
    for ax in axes:
        ax.grid(alpha=0.25, which="both")
        ax.legend()
    fig.tight_layout()
    fig.savefig(product["directory"] / "quicklook.png", dpi=140)
    plt.show()

plot_product(pilot, pilot_plan, "4096 s pilot")

## 6. Generate the full year

**This cell launches the one-year run configured above.** The pilot remains separate.
The run retains the full science duration after removing the documented guards.
A production runtime has not been benchmarked here. Keep the kernel running and leave space for the raw file.

In [ ]:
year = generate_dataset(OUTPUT_ROOT / "one_year", plan)
year

In [ ]:
year_qc = inspect_product(year, plan)
plot_product(year, plan, "First seven days of the one-year data")

## 7. Use the saved data

`instrument.h5` contains the raw simulated interferometer measurements and ranging information.
`tdi_XYZ_AET.h5` contains `t` (relative seconds), `t_lisa` (absolute TCB seconds since the LISA epoch),
and `X2`, `Y2`, `Z2`, `A2`, `E2`, `T2` in Hz. `manifest.json` records provenance and completion.
All arrays are float64. Divide channels by the recorded carrier frequency only if you deliberately want
fractional-frequency combinations; this does not convert TDI to strain.

Read selected intervals from HDF5 instead of loading the complete raw instrument file into memory.
The example below loads one day of A/E/T. Change `start_s` and `stop_s` to select another interval.

In [ ]:
start_s = 0.0
stop_s = min(86400.0, plan.duration_s)
i0 = int(start_s / plan.output_dt_s)
i1 = int(stop_s / plan.output_dt_s)
with h5py.File(year["tdi"], "r") as data:
    selected = {name: data[name][i0:i1] for name in ("t", "t_lisa", "A2", "E2", "T2")}
print({name: values.shape for name, values in selected.items()})
print("Data:", year["tdi"])
print("Provenance:", year["manifest"])
# Optional small-interval export for another tool:
# np.savez_compressed(year["directory"] / "selected_interval.npz", **selected)

## Verification and scope

The notebook checks sample counts, finite values, absolute/relative timestamps and the A/E/T transformation.
The preparation-time numerical checks compare blockwise TDI with the existing whole-record PyTDI route on a short realization,
and check the resampling filter independently. These checks establish the tested software route and numerical consistency.
They do not establish a calibrated mission noise floor, clock-correction performance, annual sensitivity, or a detection threshold.
Existing upstream messages about file-version compatibility and the default static beatnote-frequency plan remain visible;
short-run agreement does not resolve those instrument-model qualifications.
The full year is generated only when you execute section 6.

References for the upstream interfaces:
[LISA Instrument and its native scheduler](https://lisa-simulation.pages.in2p3.fr/instrument/instrument.html),
[chunked file reading](https://lisa-simulation.pages.in2p3.fr/instrument/notebooks/chunked_processing_demo.html).
The tested versions are printed above and saved per run; newer online documentation may describe newer releases.